# Fine-tuning with LoRA on synthetic data

Exercise notebook for the lesson **[Fine-tuning with LoRA on synthetic data](https://ramkumarmd.com/medical-ai/generative-ai-llms/fine-tuning-lora)** in the *Generative AI & LLMs* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** Fine-tuning adapts a pre-trained model to a specific task by training it further on examples. LoRA (low-rank adaptation) does this cheaply: it freezes the original weights and trains small add-on matrices, often less than 1% of the parameters. Fine-tuning is best for teaching a consistent format, style or narrow skill; for up-to-date facts, retrieval is usually better. Fine-tuning on patient data raises privacy risks, because models can memorise training text.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### How LoRA works

How many parameters LoRA trains for one 4,096 × 4,096 weight matrix.

In [ ]:
d = 4096   # width of one weight matrix in a mid-sized language model layer
for rank in [4, 8, 16, 64]:
    full = d * d
    lora = 2 * d * rank          # A is r x d, B is d x r
    print(f"rank {rank:>2}: LoRA trains {lora:>9,} parameters vs {full:,} ({lora / full:.2%})")

# rank  4: LoRA trains    32,768 parameters vs 16,777,216 (0.20%)
# rank  8: LoRA trains    65,536 parameters vs 16,777,216 (0.39%)
# rank 16: LoRA trains   131,072 parameters vs 16,777,216 (0.78%)
# rank 64: LoRA trains   524,288 parameters vs 16,777,216 (3.12%)

### LoRA in NumPy

Train LoRA adapters of rank 1, 2, 3 and 8 on a task that needs a rank-3 change.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
d = 64
W0 = rng.normal(0, 1 / np.sqrt(d), (d, d))                 # frozen "pre-trained" weights
U, V = rng.normal(0, 0.3, (d, 3)), rng.normal(0, 0.3, (3, d))
true_change = U @ V                                          # the new task needs a rank-3 change
X = rng.normal(size=(2000, d))
Y = X @ (W0 + true_change).T

def train_lora(rank, steps=1000, lr=0.05):
    r = np.random.default_rng(1)
    A = r.normal(0, 0.01, (rank, d))   # trained
    B = np.zeros((d, rank))            # trained; starts at zero so the model starts as W0
    for _ in range(steps):
        err = X @ (W0 + B @ A).T - Y
        grad = err.T @ X / len(X)
        B, A = B - lr * grad @ A.T, A - lr * B.T @ grad
    return np.mean(err**2), A.size + B.size

print(f"No adaptation:  error {np.mean((X @ W0.T - Y)**2):.3f}")
for rank in [1, 2, 3, 8]:
    error, n_params = train_lora(rank)
    print(f"LoRA rank {rank}:    error {error:.3f}   trained parameters {n_params} of {W0.size}")

# No adaptation:  error 1.596
# LoRA rank 1:    error 0.886   trained parameters 128 of 4096
# LoRA rank 2:    error 0.351   trained parameters 256 of 4096
# LoRA rank 3:    error 0.000   trained parameters 384 of 4096
# LoRA rank 8:    error 0.000   trained parameters 1024 of 4096

## Exercise

**Goal:** Implement and explore LoRA in NumPy, then plan a responsible clinical fine-tuning project.

**Data:** Synthetic matrices and examples generated in the notebook (Generated in the notebook)

### Step 1

Reproduce the rank comparison and plot error against rank.

In [ ]:
# Your code here


### Step 2

Add noise to the targets and use a held-out set to see whether a large rank overfits.

In [ ]:
# Your code here


### Step 3

Apply a LoRA adapter, then 'merge' it into W0 (W0 + B @ A) and confirm the outputs are identical.

In [ ]:
# Your code here


### Step 4

Optional, with a GPU: fine-tune a small open model with a LoRA adapter (Hugging Face PEFT) on the provided synthetic note-to-summary pairs, and compare with the base model plus prompting on held-out examples.

In [ ]:
# Your code here


### Step 5

Write a one-page plan covering data source, de-identification, evaluation, safety testing and who approves deployment.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/generative-ai-llms/fine-tuning-lora](https://ramkumarmd.com/medical-ai/generative-ai-llms/fine-tuning-lora). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.